# Run the OCSR benchmark on Google Colab

Runtime → Change runtime type → **T4 GPU**. Before running, add your Hugging Face token under the **key icon (Secrets)** as `HF_TOKEN` and switch on notebook access. Then run the cells in order.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Replace `YOUR-USERNAME` with your GitHub username.

In [ ]:
REPO_URL = "https://github.com/vaibhavi-tec/ocsr-benchmark"
!git clone -q $REPO_URL ocsr-benchmark
%cd ocsr-benchmark
!git log -1 --oneline

## 2. Create the Python 3.10 environment
Colab's own Python is newer than BioMiner's tested setup, so `uv` creates a separate Python 3.10 environment (about 2–3 minutes).

In [ ]:
!pip install -q uv
!uv venv -q --python 3.10 .venv
!uv pip install -q --python .venv -r requirements.txt
!.venv/bin/python -c "import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())"

## 3. Hugging Face token (for the gated MolGlyph model)
Read from Colab Secrets; it is never printed or saved.

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN set:", bool(os.environ["HF_TOKEN"]))

## 4. Download the BioVista OCSR dataset

In [ ]:
!.venv/bin/python scripts/download_data.py

## 5. Check everything works
Metric tests, then a 20-image run of both tools.

In [ ]:
!.venv/bin/python tests/test_metrics.py | tail -1
!.venv/bin/python src/run_benchmark.py --limit 20 --fresh

## 6. Full benchmark (all 8,861 images)
Progress is saved every 200 images. If Colab disconnects, rerun cells 1–4 (skip `git clone` if the folder exists) and this cell again: it resumes where it stopped (the 20 test images from step 5 are reused, not rerun).

In [ ]:
!.venv/bin/python src/run_benchmark.py

## 7. Download the results
Add the files to the `results/` folder of your GitHub repo.

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")